### `MICrONS Connectome & Functional`

> Embedding Neuronal Constraints Drive Superior Learning in Recurrent Neural Networks

| Session | Scan | Fields        |
|---------|------|---------------|
| 4       | 7    | [2, 4, 6, 8]   |
| 5       | 3    | [2, 4, 6, 8]   |
| 5       | 6    | [2, 4, 6, 8]   |
| 5       | 7    | [2, 4, 6, 8]   |
| 6       | 2    | [2, 4, 6, 8]   |
| 6       | 4    | [2, 4, 6, 8]   |
| 6       | 6    | [2, 4, 6, 8]   |
| 6       | 7    | [2, 4, 6, 8]   |
| 7       | 3    | [2, 4, 6, 8]   |
| 7       | 4    | [2, 4, 6, 8]   |
| 7       | 5    | [2, 4, 6, 8]   |
| 8       | 5    | [2, 4, 6, 8]   |
| 9       | 3    | [2, 4, 6]      |
| 9       | 4    | [2, 4, 6]      |
| 9       | 6    | [2, 4]         |

In [ ]:
session, scan, field = 6, 6, 2

##### 1. Getting started

In [ ]:
%env DJ_HOST=db.datajoint.com
%env DJ_USER=microns
%env DJ_PASS=microns2023
%env DATABASE_PREFIX=microns_

! apt update && apt install -y git graphviz >/dev/null
! pip install -qq seaborn ipykernel
! pip install -qq datajoint
! pip install -qq caveclient
! pip install -qq s3fs
! apt -y install mysql-server >/dev/null
! service mysql start
! mysql -e "ALTER USER 'root'@'localhost' IDENTIFIED WITH 'mysql_native_password' BY '';FLUSH PRIVILEGES;"

In [ ]:
!git clone https://github.com/datajoint/microns_phase3_nda
!mv microns_phase3_nda/microns_phase3 .
!rm -rf microns_phase3_nda

In [ ]:
import pandas as pd #type: ignore
import seaborn as sns #type: ignore
import numpy as np #type: ignore
from matplotlib import pyplot as plt #type: ignore
import caveclient #type: ignore
import networkx as nx #type: ignore
import matplotlib.pyplot as plt #type: ignore
import matplotlib.cm as cm #type: ignore
import matplotlib.colors as mcolors #type: ignore
from scipy.sparse import csr_matrix #type: ignore
from collections import Counter #type: ignore
import scipy.sparse as sp #type: ignore
from networkx.algorithms import community #type: ignore
from matplotlib.cm import ScalarMappable #type: ignore
from matplotlib.colors import Normalize #type: ignore
from time import sleep # type: ignore
from tqdm.notebook import tqdm as tdqm # type: ignore
from microns_phase3 import nda #type: ignore
from scipy.signal import find_peaks #type: ignore

***If you have not acquired an authentication token, see below to get a new token.You will need to link your Google account.***

In [ ]:
client = caveclient.CAVEclient()
client.auth.get_new_token()

In [ ]:
from google.colab import userdata,files
token = userdata.get('TOKEN')
client.auth.save_token(token=token, overwrite=True)

In [ ]:
# client.auth.save_token(token="PASTE_YOUR_TOKEN_HERE", overwrite=True)
client = caveclient.CAVEclient("minnie65_public")

##### 2. Querying the coregistred neurons

- **coregistration_manual_v4**: This table contains the results of manually verified coregistration. It is well-verified but contains fewer ROIs (N=15,352 root IDs, 19,181 ROIs).

- **coregistration_auto_phase3_fwd_apl_vess_combined**: This table contains the results of automated functional matching between the EM and 2-p functional data. It is not manually verified but contains more ROIs (N=84,233 ROIs). This table reconciles the following two tables that both make a best match of the registration using different techniques: `coregistration_auto_phase3_fwd` and `apl_functional_coreg_vess_fwd`.

In [ ]:
# Query the coregistration table
query_result = client.materialize.query_table('coregistration_manual_v4') #have the option 'coregistration_auto_phase3_fwd_apl_vess_combined'
query_result = query_result[(query_result['session'] == session) & (query_result['scan_idx'] == scan) & (query_result['field'] == field)]

`Column descriptions`

<b>id:</b> a unique identifier for this row

<b>valid:</b> internal check, uniformly ‘t’

<b>pt_supervoxel_id:</b> the ID of the supervoxel from the watershed segmentation that is under the pt_position

<b>pt_root_id:</b> the ID of the segment/root_id under the pt_position from the Proofread Segmentation (v343).

<b>session:</b> the ID indicating the imaging period for the mouse

<b>scan_idx:</b> the index of the scan within the imaging session

<b>unit_id:</b> the ID of the functional ROI (unique per scan)

<b>pt_position_{x,y,z}:</b> the location in 4,4,40 nm voxels at a cell body for the cell

<b>residual:</b> the Euclidean distance between the EM nucleus centroid and the matched functional centroid after the AIBS EM -> 2P coregistration transform

<b>separation:</b> For a matched EM nucleus to a functional unit the separation score is computed as the difference between the residual of the matched pair and the residual of the nearest EM neuronal nucleus that was not matched to the unit.

> More confident matches will have a smaller residual and larger separation

In [ ]:
root_ids = query_result['pt_root_id'].values
positions = query_result['pt_position'].values
target_ids = query_result['target_id'].values
unit_ids = query_result['unit_id'].values

##### 3. Connectome & Functional

In [ ]:
def get_connectivity_matrix(target_list, contain_self_loop=True):
    target_list = list(set(target_list)) # Remove duplicates
    num_neurons = len(target_list)

    # Initialize a sparse matrix in LIL (List of Lists) format for efficient row-wise modification
    connectivity_matrix = sp.lil_matrix((num_neurons, num_neurons), dtype=np.int64)

    for i, root_id in enumerate(target_list):
        sleep(0.5)

        # Query synaptic partners
        output = client.materialize.synapse_query(pre_ids=root_id).get('post_pt_root_id', []).to_list()
        output_counter = Counter(output)  # Precompute connection counts

        for j, root_id2 in enumerate(target_list):
            if i == j and not contain_self_loop:
                continue  # Skip self-loops if disabled
            count = output_counter.get(root_id2, 0)
            if count > 0:
                connectivity_matrix[i, j] = count  # Assign nonzero values

    # Convert to CSR format for efficient computation and storage
    return connectivity_matrix.tocsr()

In [ ]:
connectome = get_connectivity_matrix(root_ids, contain_self_loop=True)

In [ ]:
def get_spikes(session, scan, unit_id):
    unit_key = {'session': session, 'scan_idx': scan, 'unit_id': unit_id}

    # Fetch data
    nframes, fps = (nda.Scan & unit_key).fetch1('nframes', 'fps')
    time_axis = np.arange(nframes) / fps
    spike_trace = (nda.Activity & unit_key).fetch1('trace')
    calcium_trace = (nda.ScanUnit * nda.Fluorescence & unit_key).fetch1('trace')
    peaks = find_peaks(spike_trace, threshold=0.01)
    spike_times = time_axis[peaks[0]]

    return spike_times, peaks, time_axis, calcium_trace, spike_trace

In [ ]:
# save connectome
connectome_df = pd.DataFrame.sparse.from_spmatrix(connectome)
connectome_df.to_csv(f'connectome_{session}_{scan}_{field}.csv')
files.download(f'connectome_{session}_{scan}_{field}.csv')

In [ ]:
# save SPIKES
SPIKES = []
for unit_id in tdqm(unit_ids):
    sleep(0.5)
    spike, _, _, _, _ = get_spikes(session, scan, unit_id)
    SPIKES.append(spike)

SPIKES = np.array(SPIKES, dtype=object)
np.save(f'spikes_{session}_{scan}_{field}.npy', SPIKES)
files.download(f'spikes_{session}_{scan}_{field}.npy')

In [ ]:
# save root_ids, ...
root_ids_np = np.array(root_ids)
positions_np = np.array(positions)
target_ids_np = np.array(target_ids)
unit_ids_np = np.array(unit_ids)

np.save(f'root_ids_{session}_{scan}_{field}.npy', root_ids_np)
np.save(f'positions_{session}_{scan}_{field}.npy', positions_np)
np.save(f'target_ids_{session}_{scan}_{field}.npy', target_ids_np)
np.save(f'unit_ids_{session}_{scan}_{field}.npy', unit_ids_np)

files.download(f'root_ids_{session}_{scan}_{field}.npy')
files.download(f'positions_{session}_{scan}_{field}.npy')
files.download(f'target_ids_{session}_{scan}_{field}.npy')
files.download(f'unit_ids_{session}_{scan}_{field}.npy')

---